### WEEK 6: FEATURE ENGINEERING AND DATA PREPROCESSING

#### Project Title
Spotify Music Trends and Audio Features Analysis Using Data Science and Visualization

#### Course
24ADI204 – Data Science and Visualization

#### Team
DSV Team 9

#### Aim
To create useful Spotify features, encode categorical attributes, correct skewness, and scale numerical features for further machine learning analysis.

#### Objectives
1. Load the cleaned Spotify dataset from Week 3.
2. Create song duration in minutes and album release year.
3. Select relevant numerical and categorical attributes.
4. Check and handle any remaining missing values.
5. Identify skewed features and apply the Yeo–Johnson transformation.
6. Encode categorical variables using one-hot encoding.
7. Apply standard scaling and Min-Max normalization.
8. Combine processed features and prepare the popularity target.

#### Tools Used
- Python
- Pandas and NumPy
- Matplotlib
- Scikit-learn
- Jupyter Notebook

### 1. Importing Required Libraries

Pandas and NumPy are used for data manipulation and numerical calculations. Matplotlib is used to visualize distributions. Scikit-learn provides tools for missing-value imputation, categorical encoding, skewness transformation, and feature scaling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

### 2. Loading the Cleaned Spotify Dataset

The dataset cleaned in Week 3 is loaded from `spotify_cleaned.csv`. Using the cleaned file maintains continuity between the weekly experiments. The first records and dataset dimensions are displayed to confirm successful loading.

In [ ]:

import pandas as pd
from pathlib import Path

file_path = Path("DSV_Team-9_Cleaned_Dataset.csv")

if not file_path.exists():
    print("Dataset not found in current folder.")
    print("Current folder:", Path.cwd())

    file_path = Path(
        input("Enter full path of your cleaned CSV file: ").strip().strip('"')
    )

if not file_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {file_path}")

df = pd.read_csv(file_path, low_memory=False)

print("Cleaned dataset loaded successfully!")
print("Dataset Shape:", df.shape)


### 3. Inspecting Dataset Structure

The available columns and their data types are examined using `df.columns` and `df.info()`. This helps identify which Spotify attributes can be used for feature engineering and preprocessing.

In [ ]:
print("Columns in the dataset:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

### 4. Feature Engineering

Feature engineering creates new useful variables from existing Spotify metadata. In this experiment, song duration is converted from milliseconds to minutes, and the release year is extracted from the album release date.

- duration_minutes from song duration in milliseconds.
- release_year from the album release date.

#### Before Feature Engineering

The original song name, duration, release date, and popularity fields are displayed to understand the values before creating new attributes.

In [ ]:
before_columns = [
    c for c in [
        "name",
        "duration_ms.x",
        "album.release_date",
        "popularity"
    ]
    if c in df.columns
]

display(df[before_columns].head(10))

#### Creating New Features

The `duration_minutes` attribute is calculated by dividing duration in milliseconds by 60,000. The `release_year` attribute is extracted using Pandas date conversion. A preview is displayed to compare the newly created features with their source columns.

In [ ]:
df_features = df.copy()

if "duration_ms.x" in df_features.columns:
    duration_col = "duration_ms.x"
elif "duration_ms.y" in df_features.columns:
    duration_col = "duration_ms.y"
else:
    duration_col = None

if duration_col:
    df_features["duration_minutes"] = (
        pd.to_numeric(df_features[duration_col], errors="coerce") / 60000
    )

if "album.release_date" in df_features.columns:
    df_features["release_year"] = pd.to_datetime(
        df_features["album.release_date"],
        errors="coerce"
    ).dt.year

after_columns = [
    c for c in [
        "name",
        duration_col,
        "album.release_date",
        "duration_minutes",
        "release_year",
        "popularity"
    ]
    if c is not None and c in df_features.columns
]

print("AFTER FEATURE ENGINEERING")
display(df_features[after_columns].head(10))

### 5. Selecting Numerical and Categorical Features

Numerical audio features such as danceability, energy, tempo, and valence are selected for numerical preprocessing. Categorical features such as explicit status, album type, musical key, mode, and time signature are selected for encoding. Only columns available in the dataset are included.

In [ ]:
numeric_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "duration_minutes",
    "release_year"
]

categorical_features = [
    "explicit",
    "album.album_type",
    "key",
    "mode",
    "time_signature"
]

numeric_features = [
    c for c in numeric_features if c in df_features.columns
]

categorical_features = [
    c for c in categorical_features if c in df_features.columns
]

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

print("\nNumerical features before processing:")
display(df_features[numeric_features].head())

print("\nCategorical features before processing:")
display(df_features[categorical_features].head())

### 6. Checking Missing Values Before Imputation

Missing-value counts are calculated for the selected numerical and categorical features. This check determines whether additional imputation is needed before applying transformations.

In [ ]:
selected_features = numeric_features + categorical_features

print("Missing values BEFORE imputation:")
display(
    df_features[selected_features].isnull().sum()
    .rename("Missing count")
    .to_frame()
)

### 7. Handling Remaining Missing Values

Median imputation is used for numerical features because it is less sensitive to extreme values. Most-frequent-value imputation is used for categorical features. The processed feature tables and remaining missing-value counts are displayed for verification.

In [ ]:
X_numeric = df_features[numeric_features].apply(
    pd.to_numeric, errors="coerce"
)

numeric_imputer = SimpleImputer(strategy="median")

X_numeric_imputed = pd.DataFrame(
    numeric_imputer.fit_transform(X_numeric),
    columns=numeric_features,
    index=df_features.index
)

if categorical_features:
    X_categorical = df_features[categorical_features].copy()

    categorical_imputer = SimpleImputer(strategy="most_frequent")

    X_categorical_imputed = pd.DataFrame(
        categorical_imputer.fit_transform(X_categorical),
        columns=categorical_features,
        index=df_features.index
    )
else:
    X_categorical_imputed = pd.DataFrame(index=df_features.index)

print("Numerical features AFTER imputation:")
display(X_numeric_imputed.head())

print("Categorical features AFTER imputation:")
display(X_categorical_imputed.head())

print(
    "Remaining numerical missing values:",
    X_numeric_imputed.isnull().sum().sum()
)

print(
    "Remaining categorical missing values:",
    X_categorical_imputed.isnull().sum().sum()
)

### 8. Analyzing Skewness Before Transformation

Skewness describes the asymmetry of a numerical distribution. The skewness coefficient is calculated for each numerical feature, and histograms are plotted to inspect their distributions before correction.

In [ ]:
skew_before = X_numeric_imputed.skew()

print("Skewness BEFORE Yeo-Johnson:")
display(skew_before.rename("Before").to_frame())

X_numeric_imputed.hist(figsize=(14, 10), bins=25)
plt.suptitle("Numerical Feature Distributions Before Transformation")
plt.tight_layout()
plt.show()

### 9. Correcting Skewness Using Yeo–Johnson

Features with absolute skewness greater than 1 are selected for transformation. The Yeo–Johnson method is applied to reduce skewness and can work with zero and negative values. Skewness values before and after transformation are compared.

In [ ]:
skewed_features = [
    c for c in numeric_features
    if abs(skew_before[c]) > 1
    and X_numeric_imputed[c].nunique() > 1
]

print("Features selected for transformation:")
print(skewed_features)

X_skew_fixed = X_numeric_imputed.copy()

if skewed_features:
    yeo_johnson = PowerTransformer(
        method="yeo-johnson",
        standardize=False
    )

    X_skew_fixed[skewed_features] = yeo_johnson.fit_transform(
        X_numeric_imputed[skewed_features]
    )

skew_after = X_skew_fixed.skew()

print("\nSkewness BEFORE and AFTER:")
display(pd.DataFrame({
    "Before": skew_before,
    "After": skew_after
}))

#### Comparing Distributions Before and After Transformation

Histograms are displayed side by side for up to four transformed features. This helps visually evaluate whether the Yeo–Johnson transformation made the distributions less skewed.

In [ ]:
for c in skewed_features[:4]:
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.hist(X_numeric_imputed[c], bins=25)
    plt.title(c + " - Before")

    plt.subplot(1, 2, 2)
    plt.hist(X_skew_fixed[c], bins=25)
    plt.title(c + " - After")

    plt.tight_layout()
    plt.show()

### 10. Encoding Categorical Variables

Machine learning algorithms generally require numerical input. The categorical features are first previewed in their original form and then converted into numerical indicator columns using one-hot encoding.

In [ ]:
print("Categorical variables BEFORE encoding:")
display(X_categorical_imputed.head(10))

#### Applying One-Hot Encoding

The `OneHotEncoder` creates separate binary columns for each categorical value. `handle_unknown="ignore"` allows unseen categories to be handled during future transformations.

In [ ]:
if categorical_features:
    encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

    encoded_array = encoder.fit_transform(X_categorical_imputed)

    encoded_columns = encoder.get_feature_names_out(
        categorical_features
    )

    X_encoded = pd.DataFrame(
        encoded_array,
        columns=encoded_columns,
        index=df_features.index
    )
else:
    X_encoded = pd.DataFrame(index=df_features.index)

print("Categorical variables AFTER encoding:")
display(X_encoded.head(10))

print("Original categorical columns:", len(categorical_features))
print("Encoded columns:", X_encoded.shape[1])

### 11. Standard Scaling

Standard scaling transforms numerical features to have a mean close to 0 and a standard deviation close to 1. The original feature values and summary statistics are displayed before scaling so they can be compared with the standardized results.

In [ ]:
print("Numerical values BEFORE standard scaling:")
display(X_skew_fixed.head())

display(
    X_skew_fixed.describe().T[["mean", "std", "min", "max"]]
)

#### Applying Standard Scaling

The `StandardScaler` is fitted to the numerical feature table and used to transform its values. The scaled data and summary statistics are displayed to confirm the change in mean and standard deviation.

In [ ]:
scaler = StandardScaler()

X_standard_scaled = pd.DataFrame(
    scaler.fit_transform(X_skew_fixed),
    columns=numeric_features,
    index=df_features.index
)

print("Numerical values AFTER standard scaling:")
display(X_standard_scaled.head())

display(
    X_standard_scaled.describe().T[["mean", "std", "min", "max"]]
)

### 12. Min-Max Normalization

Min-Max normalization rescales numerical values to the range from 0 to 1. This provides an alternative to standard scaling and allows numerical features with different units to be compared on a common scale.

In [ ]:
print("Numerical values BEFORE normalization:")
display(X_skew_fixed.head())

#### Applying Min-Max Normalization

The `MinMaxScaler` is applied to the numerical feature table. The resulting values, minimums, and maximums are displayed to verify the normalization range.

In [ ]:
normalizer = MinMaxScaler()

X_normalized = pd.DataFrame(
    normalizer.fit_transform(X_skew_fixed),
    columns=numeric_features,
    index=df_features.index
)

print("Numerical values AFTER normalization:")
display(X_normalized.head())

print("Minimum values after normalization:")
display(X_normalized.min().to_frame("Minimum"))

print("Maximum values after normalization:")
display(X_normalized.max().to_frame("Maximum"))

### 13. Combining Processed Features

The standardized numerical features and one-hot-encoded categorical features are previewed before they are combined into a single input feature table.

In [ ]:
print("Standardized numerical features:")
display(X_standard_scaled.head())

print("Encoded categorical features:")
display(X_encoded.head())

#### Creating the Final Processed Feature Table

The numerical and encoded categorical features are combined column-wise using `pd.concat()`. The final dimensions, first records, and missing-value count are displayed to verify that the input feature table is ready for further analysis.

In [ ]:
X_processed = pd.concat(
    [X_standard_scaled, X_encoded],
    axis=1
)

print("FINAL PROCESSED FEATURES")
print("Shape:", X_processed.shape)

display(X_processed.head())

print("Total missing values:", X_processed.isnull().sum().sum())

### 14. Preparing the Popularity Target

Spotify popularity is converted to a numerical target variable. Rows with missing popularity values are excluded from the final feature and target tables to ensure that they remain aligned.

In [ ]:
y = pd.to_numeric(
    df_features["popularity"],
    errors="coerce"
)

valid_rows = y.notna()

X_processed = X_processed.loc[valid_rows].copy()
y = y.loc[valid_rows].copy()

print("Final input features shape:", X_processed.shape)
print("Popularity target shape:", y.shape)

print("\nInput feature preview:")
display(X_processed.head())

print("\nPopularity target preview:")
display(y.head())


### 15. Exporting the Processed Dataset

The processed numerical and categorical features are combined with the popularity target variable.

The final dataset is exported in CSV and Excel formats for further analysis and future use.

The exported dataset contains standardized numerical features, one-hot encoded categorical features, and the popularity target.


In [ ]:

from pathlib import Path

# Combine processed features with target variable
final_dataset = X_processed.copy()
final_dataset["popularity"] = y

# Save files in the current Jupyter folder
output_folder = Path.cwd()

csv_path = output_folder / "DSV_Team-9_Week-6_Processed_Dataset.csv"
excel_path = output_folder / "DSV_Team-9_Week-6_Processed_Dataset.xlsx"

# Export CSV
final_dataset.to_csv(csv_path, index=False)

# Export Excel
final_dataset.to_excel(excel_path, index=False)

print("Processed dataset exported successfully!")
print("Dataset Shape:", final_dataset.shape)
print("CSV File:", csv_path)
print("Excel File:", excel_path)

display(final_dataset.head())


### 16. Conclusion

Feature engineering and data preprocessing were performed on the Spotify dataset. New duration and release-year attributes were created, remaining missing values were addressed, and skewed numerical features were transformed using Yeo–Johnson where appropriate. Categorical attributes were one-hot encoded, and numerical features were processed using standard scaling and Min-Max normalization. Finally, the standardized numerical features and encoded categorical features were combined, and Spotify popularity was prepared as the target variable.

#### Result

A processed feature table and corresponding popularity target were prepared for subsequent machine learning or feature-selection activities.

**Note:** For predictive modeling, imputation, transformation, encoding, and scaling should be fitted on training data only to avoid data leakage.